# 2. Resultados experimentais

Lê o que foi gravado por `experiment`, `ensemble`, `compare`, `train-final`, `explain` e `drift`.
Escolha a base em `DATASET`: `creditcard` (ULB), `paysim_sem_saldo` ou `paysim`.

As tabelas e figuras numeradas para o texto ficam em `reports/consolidado/` (`python -m fraude consolidate`), exibidas na última seção.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Image, Markdown, display
from fraude.consolidate import selections, _key_label
from fraude.evaluation import savings_vs_best_trivial
pd.set_option('display.max_columns', 40)

DATASET = 'creditcard'
REPORTS = ROOT / 'reports' / DATASET
KIND = {'creditcard': 'creditcard', 'paysim': 'paysim', 'paysim_sem_saldo': 'paysim_restrito'}[DATASET]
def show(path):
    if Path(path).exists():
        display(Image(path))

In [ ]:
exp = REPORTS / 'experiments'
info = json.loads((exp / 'dataset.json').read_text(encoding='utf-8'))
info

## Tabela comparativa

Métricas no conjunto de teste, com limiar escolhido pela função de custo (cenário moderado, FN:FP = 20:1) nas predições *out-of-fold* do treino.

In [ ]:
table = pd.read_csv(exp / 'results.csv')
table.sort_values('auc_pr', ascending=False)

## AUC-PR por algoritmo × balanceamento

In [ ]:
show(exp / 'figures' / 'auc_pr_heatmap.png')

## Curvas precisão-revocação e ROC

A curva ROC tende a parecer otimista em bases desbalanceadas (Saito; Rehmsmeier, 2015).

In [ ]:
show(exp / 'figures' / 'pr_curves.png')
show(exp / 'figures' / 'roc_curves.png')

## Função de custo e escolha do limiar

A economia é comparada também com a **melhor política trivial** (a mais barata entre aprovar e bloquear tudo). Quando a fraude custa muito mais que um bloqueio indevido — custo pelo valor no PaySim —, o limiar ótimo cai para perto de zero e "economia vs. aprovar tudo" engana.

In [ ]:
for s in ['conservador', 'moderado', 'agressivo', 'valor_transacao']:
    show(exp / 'figures' / f'cost_curves_{s}.png')
show(exp / 'figures' / 'confusion_moderado.png')

In [ ]:
records = json.loads((exp / 'results.json').read_text(encoding='utf-8'))
pd.DataFrame([{'model': r['model'], 'balancing': r['balancing'], 'cenario': s, **{
    'limiar': c['threshold'], 'custo': c['cost'], 'aprovar_tudo': c['cost_approve_all'],
    'bloquear_tudo': c['cost_block_all'], 'economia_vs_aprovar_%': c['savings_vs_approve_all_pct'],
    'economia_vs_melhor_trivial_%': savings_vs_best_trivial(c)}}
    for r in records for s, c in r['test_cost'].items()]).pivot_table(
    index=['model', 'balancing'], columns='cenario', values='economia_vs_melhor_trivial_%').round(1)

## Custo computacional

In [ ]:
show(exp / 'figures' / 'tradeoff_latency_aucpr.png')
table[['model','balancing','train_seconds','latency_mean_ms','latency_p95_ms']].sort_values('latency_mean_ms')

## Duplicatas

Transações repetidas (mesmos atributos e rótulo, ignorando o instante) são removidas antes da separação treino/teste. `reports/creditcard_v1_com_duplicatas` guarda a execução anterior.

In [ ]:
v1_path = ROOT / 'reports' / 'creditcard_v1_com_duplicatas' / 'experiments' / 'results.csv'
print('removidas:', info.get('duplicates_removed'), '| fraudes:', info.get('duplicate_frauds_removed'))
if DATASET == 'creditcard' and v1_path.exists():
    v1 = pd.read_csv(v1_path)[['model', 'balancing', 'auc_pr']]
    both = table[['model', 'balancing', 'auc_pr']].merge(v1, on=['model', 'balancing'],
                                                        suffixes=('_sem_dup', '_com_dup'))
    both['diferenca'] = both['auc_pr_sem_dup'] - both['auc_pr_com_dup']
    display(both.sort_values('auc_pr_sem_dup', ascending=False).round(4))
    print('diferença média no teste:', round(both['diferenca'].mean(), 4))

## Busca ampliada e ensembles

In [ ]:
for d in ['experiments_busca50', 'experiments_ensemble']:
    p = REPORTS / d / 'results.csv'
    if p.exists():
        display(Markdown(f'### {d}')); display(pd.read_csv(p))

## Comparação estatística

CV repetida 5×3; Friedman + Nemenyi; teste t corrigido de Nadeau-Bengio contra a melhor (Holm).

Escolha final (regra revisada): entre as não significativamente piores que a melhor **e** a no máximo 0,01 de AUC-PR dela, a de maior AUC-PR entre as mais rápidas (latências até 0,5 ms acima da mínima contam como empate). A regra original — só "a mais rápida entre as equivalentes" — escolhia na ULB um modelo 0,033 pior para ganhar 0,5 ms, porque com 15 partições o teste tem pouco poder. A tabela abaixo mostra o que cada regra escolhe.

In [ ]:
cmp = REPORTS / 'comparacao'
if cmp.exists():
    display(Markdown((cmp / 'summary.md').read_text(encoding='utf-8')))
    show(cmp / 'ranking_cd.png')
    comp = pd.read_csv(cmp / 'comparison.csv')
    t = comp.set_index('config')
    display(pd.DataFrame([{'criterio': k, 'configuracao': _key_label(v),
                           **t.loc[v, ['auc_pr_mean', 'p_holm', 'latency_mean_ms', 'test_auc_pr']]}
                          for k, v in selections(comp).items()]))
else:
    print('Sem comparação estatística para esta base.')

## Modelo final e calibração

In [ ]:
meta = json.loads((ROOT / 'artifacts' / f'model_{KIND}.json').read_text(encoding='utf-8'))
print(meta['model'], meta['balancing'], '| calibração:', meta['calibration']['method'],
      '| limiar:', round(meta['threshold'], 4))
display(pd.Series(meta['test_metrics']).to_frame('teste'))
display(pd.DataFrame({k: v for k, v in meta['calibration'].items() if k.startswith('test')}))
show(REPORTS / 'calibracao' / 'reliability.png')

## Explicabilidade (SHAP e LIME)

In [ ]:
ex = REPORTS / 'explainability'
summary = json.loads((ex / 'explainability.json').read_text(encoding='utf-8'))
display(pd.DataFrame(summary['global_top10']))
show(ex / 'shap_beeswarm.png')
for case, c in summary['cases'].items():
    print(case, round(c['probability'], 4), '| sobreposição SHAP×LIME top5:', c['top5_overlap'])
    show(ex / f'shap_waterfall_{case}.png')
    show(ex / f'lime_{case}.png')

## Concept drift

In [ ]:
for sub in ['drift', 'drift_atraso1']:
    p = REPORTS / sub / 'drift.csv'
    if not p.exists():
        continue
    drift = pd.read_csv(p)
    display(Markdown(f'### {sub}'))
    display(drift.pivot(index='window', columns='strategy', values='auc_pr').round(3))
    display(drift[drift.strategy == 'estatico'][['window', 'n_fraud', 'psi_score', 'psi_amount']])
show(REPORTS / 'drift' / 'drift.png')

## O atalho do simulador PaySim

No PaySim o agente fraudador transfere sempre o saldo inteiro da conta. Com os saldos da origem todos os algoritmos chegam a AUC-PR ≈ 1; por isso o cenário `paysim_sem_saldo` os remove.

In [ ]:
atalho = ROOT / 'reports' / 'paysim' / 'eda' / 'atalho_simulador.json'
if atalho.exists():
    sc = json.loads(atalho.read_text(encoding='utf-8'))
    display(pd.DataFrame(sc['rules']).T.round(1))

## Resultados consolidados (todas as bases)

Gerado por `python -m fraude consolidate`; tabelas em CSV e figuras numeradas em `reports/consolidado/`.

In [ ]:
doc = ROOT / 'reports' / 'consolidado' / 'resultados.md'
if doc.exists():
    text = doc.read_text(encoding='utf-8').replace('](figuras/', '](../reports/consolidado/figuras/')
    display(Markdown(text))